# KALIA v0.2.0 — the missing fifth task

The final evaluation returned PIQA, ARC-Easy, HellaSwag and WinoGrande.
LAMBADA raised a loading-script error and was never measured, which leaves
**S-A unadjudicable** — the rule is about *no* task regressing by more than
1.0 pp, and one of the five has no number.

The fix is narrow on purpose. The dataset is present (6 parquet files, the
standard 5,153 examples, one `text` column); only lm-eval's requested loader
path is dead, because the repo still carries a loading script that modern
`datasets` refuses to execute. So `load_dataset` is wrapped to resolve that
one repo to its own parquet files, and **lm-eval still does the scoring** — a
hand-rolled LAMBADA scorer would give a number that is not comparable with
v0.1.2's 23.0, and S-A would be comparing two different measurements.

CPU, no quota.

In [ ]:
!pip install -q "lm-eval==0.4.9" "transformers<5" tiktoken pyyaml datasets
import glob, os, shutil, sys
root = sorted(glob.glob('/kaggle/input/datasets/subhajitlucky/kalia-code-dev'))[0]
for f in ['eval_bench.py', 'kalia_lm.py', 'model.py', 'data.py',
          'eval_reversibility.py', 'lambada_loader.py']:
    shutil.copy(f'{root}/{f}', '/kaggle/working/')
os.chdir('/kaggle/working')
ckpt = sorted(glob.glob(
    '/kaggle/input/notebooks/subhajitlucky/kalia-train-v020/**/ckpt.pt', recursive=True))
assert ckpt, 'attach the kalia-train-v020 kernel output'
CKPT = ckpt[0]
print('checkpoint:', CKPT)

In [ ]:
# The loader fix lives in lambada_loader.py, in the repo, with tests -- not
# inline in this notebook. Two versions of this kernel failed that way: v1
# globbed all six parquet files (30,918 rows = six copies of the task), and
# v2 filtered on '/en/' believing it selected one file when it matched two.
# The module pins the config to what lm-eval 0.4.9 actually requests and
# asserts the standard 5,153 rows before any scoring happens.
from huggingface_hub import HfApi
from lambada_loader import REPO, config_parquets, load_lambada

files = HfApi().list_repo_files(REPO, repo_type='dataset')
print('configs in repo:', sorted(config_parquets(files)))
ds = load_lambada(files)
print('rows:', len(ds), '| columns:', list(ds.column_names))
print('sample:', repr(ds[0]['text'][:150]))

In [ ]:
import subprocess
rc = subprocess.run(
    [sys.executable, 'eval_bench.py',
     '--ckpt', CKPT,
     '--tasks', 'lambada_openai',
     '--out', '/kaggle/working/bench_lambada.json',
     '--limit', '500'],
    capture_output=True, text=True,
)
print(rc.stdout[-3000:])
if rc.returncode != 0:
    print('STDERR:', rc.stderr[-4000:])
# Version 4 ran eval_bench, it failed on a missing module, and the cell
# carried on -- so the failure surfaced two cells later as a bare
# FileNotFoundError on the results file, with the real cause buried in
# captured output nobody printed. Fail here, where the cause is.
assert rc.returncode == 0, f'eval_bench failed (exit {rc.returncode}); see STDERR above'
import os
assert os.path.exists('/kaggle/working/bench_lambada.json'), (
    'eval_bench exited 0 but wrote no results file')
print('results file written')

In [ ]:
import json
res = json.load(open('/kaggle/working/bench_lambada.json'))
print(json.dumps(res, indent=2))
m = res['lambada_openai']
got = m.get('acc,none', float('nan')) * 100
REF, ALLOW = 23.0, 1.0
d = got - REF
print()
print('=== S-A, fifth task ===')
print(f'  v0.1.2  {REF:.1f}')
print(f'  v0.2.0  {got:.1f}   ({d:+.2f} vs v0.1.2)')
print(f"  S-A: {'FAIL' if d < -ALLOW else 'ok'} (allowance {ALLOW} pp)")
print()
print('=== S-A, all five ===')
rows = [
    '  PIQA        61.4 ->  63.8   +2.40  ok',
    '  ARC-Easy    45.8 ->  42.0   -3.80  FAIL',
    '  HellaSwag   36.8 ->  39.8   +3.00  ok',
    '  WinoGrande  50.2 ->  50.8   +0.60  ok',
]
rows.append('  LAMBADA     %.1f ->  %.1f   %+.2f  %s' % (REF, got, d, 'FAIL' if d < -ALLOW else 'ok'))
for line in rows:
    print(line)
print()
print('Standard errors on these tasks at limit=500 are about 2.0-2.2 pp (D45), so read')
print('the +/-0.6 as noise and the -3.80 as the one signal that is not.')

Whatever LAMBADA shows, two things are already settled and should not be
re-litigated when this lands:

- **P-B passes.** Deterministic val loss **2.8248** against v0.1.2's 3.0533 on
  the identical 100-batch protocol — a **+0.2285 nat** improvement, 4.6× the
  registered 0.05 bar. bpB 0.9244 vs 0.9992.
- **S-A fails on ARC-Easy** at −3.80 pp against a 1.0 pp allowance, which is
  registered symmetrically with P-B and is reported as it stands.

The interim step-3,470 benchmark showed four of five tasks regressing. The final
checkpoint reversed that on three of them: PIQA −0.2 → +2.40, WinoGrande
−0.8 → +0.60, HellaSwag +1.8 → +3.00. Only ARC-Easy stayed negative. Whatever
conclusion is drawn about the corpus, it should be drawn from the final
checkpoint, not the interim one.